# Heart Disease Prediction Using Logistic Regression
**Google Colab | Python | Scikit-learn**

## Project objective
Build an educational binary-classification model to predict the `TenYearCHD` target from the Framingham-style heart-risk dataset.  
`0` = no recorded CHD event within the 10-year outcome period; `1` = recorded CHD event.

> **Important:** This is a learning exercise, not a medical diagnostic or clinical risk tool. Do not use its output to make healthcare decisions.

## 1. Import libraries
Run the cells from top to bottom. The notebook uses a pipeline so imputation and scaling are fitted only on the training data, reducing data leakage.

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from google.colab import files
from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, classification_report, roc_auc_score, roc_curve
)
import joblib

print("Libraries imported successfully")

ModuleNotFoundError: No module named 'pandas'

## 2. Upload and load your CSV
Upload the original `.csv` file from your computer. If the file you have is saved as `.txt` but contains comma-separated data with a header row, it can also be selected.

In [ ]:
uploaded = files.upload()
file_name = next(iter(uploaded))
df = pd.read_csv(file_name, na_values=["NA", "N/A", ""])
df.columns = df.columns.str.strip()

print("Loaded:", file_name)
print("Shape:", df.shape)
display(df.head())

## 3. Verify the dataset
Check dimensions, column names, data types, summary statistics, missing values, duplicates, and target labels before modelling.

In [ ]:
print("Shape:", df.shape)
print("\nColumns:", df.columns.tolist())
print("\nData types:")
print(df.dtypes)
print("\nSummary:")
display(df.describe(include="all").T)
print("\nMissing values:")
display(df.isna().sum().sort_values(ascending=False))
print("\nExact duplicate rows:", df.duplicated().sum())

target = "TenYearCHD"
assert target in df.columns, f"Missing target column: {target}"
print("\nTarget values:", sorted(df[target].dropna().unique()))
print("\nTarget counts:")
display(df[target].value_counts(dropna=False))

## 4. Validate columns and clean rows
The expected dataset has 15 predictor columns and the `TenYearCHD` target. The code checks the target is binary, removes rows with a missing target (because supervised learning needs a known label), and removes exact duplicate rows.

Do not automatically delete unusual clinical measurements just because they look extreme. Investigate them against the dataset documentation first.

In [ ]:
expected_features = [
    "male", "age", "education", "currentSmoker", "cigsPerDay",
    "BPMeds", "prevalentStroke", "prevalentHyp", "diabetes",
    "totChol", "sysBP", "diaBP", "BMI", "heartRate", "glucose"
]
missing_cols = [c for c in expected_features + [target] if c not in df.columns]
if missing_cols:
    raise ValueError(f"Expected columns missing from CSV: {missing_cols}")

# Keep only the expected project columns, in a consistent order.
df = df[expected_features + [target]].copy()

# Convert fields to numeric; unexpected text becomes NaN and is reported.
for col in df.columns:
    df[col] = pd.to_numeric(df[col], errors="coerce")

before = len(df)
df = df.dropna(subset=[target])
df = df.drop_duplicates().reset_index(drop=True)

if not set(df[target].unique()).issubset({0, 1}):
    raise ValueError("TenYearCHD must contain only 0 and 1 after cleaning.")

df[target] = df[target].astype(int)

print("Rows before cleaning:", before)
print("Rows after removing missing target / exact duplicates:", len(df))
print("Remaining missing values by column:")
display(df.isna().sum())
print("Cleaned target distribution:")
display(df[target].value_counts())

## 5. Explore the target distribution
The outcome may be imbalanced. This is why accuracy alone is not enough; inspect recall, precision, F1-score, and ROC-AUC as well.

In [ ]:
plt.figure(figsize=(6, 4))
sns.countplot(x=target, data=df)
plt.title("TenYearCHD class distribution")
plt.xlabel("TenYearCHD (0 = no event, 1 = event)")
plt.ylabel("Number of records")
plt.show()

## 6. Separate features and target; split the data
Use an 80% training / 20% testing split. `stratify=y` preserves approximately the same class proportions in both sets. The test set is held aside for final evaluation.

In [ ]:
X = df[expected_features]
y = df[target]

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("X_train:", X_train.shape)
print("X_test :", X_test.shape)
print("Training target proportions:")
print(y_train.value_counts(normalize=True))
print("Testing target proportions:")
print(y_test.value_counts(normalize=True))

## 7. Preprocess and train Logistic Regression
Median imputation fills missing predictor values. StandardScaler standardizes feature scales. Both are fitted inside the pipeline on training data only.

`class_weight="balanced"` adjusts class weights to account for class imbalance. Compare results with and without this setting if you want to study its effect.

In [ ]:
model = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
    ("classifier", LogisticRegression(
        max_iter=2000,
        class_weight="balanced",
        random_state=42
    ))
])

model.fit(X_train, y_train)
print("Model training complete.")

## 8. Predict and evaluate on the test set
The default decision threshold for `predict()` is 0.5. For a health-related classification problem, false negatives and false positives both matter; interpret metrics carefully.

In [ ]:
y_pred = model.predict(X_test)
y_prob = model.predict_proba(X_test)[:, 1]

print("Accuracy :", accuracy_score(y_test, y_pred))
print("Precision:", precision_score(y_test, y_pred, zero_division=0))
print("Recall   :", recall_score(y_test, y_pred, zero_division=0))
print("F1-score :", f1_score(y_test, y_pred, zero_division=0))
print("ROC-AUC  :", roc_auc_score(y_test, y_prob))

print("\nClassification report:")
print(classification_report(
    y_test, y_pred,
    labels=[0, 1],
    target_names=["No recorded CHD event", "Recorded CHD event"],
    zero_division=0
))

## 9. Confusion matrix and ROC curve
Rows in the confusion matrix are actual classes; columns are predicted classes. The ROC curve shows the true-positive rate against the false-positive rate at different thresholds.

In [ ]:
cm = confusion_matrix(y_test, y_pred, labels=[0, 1])
plt.figure(figsize=(6, 4))
sns.heatmap(
    cm, annot=True, fmt="d", cmap="Blues",
    xticklabels=["Predicted 0", "Predicted 1"],
    yticklabels=["Actual 0", "Actual 1"]
)
plt.title("Confusion Matrix")
plt.ylabel("Actual")
plt.xlabel("Predicted")
plt.show()

fpr, tpr, thresholds = roc_curve(y_test, y_prob)
plt.figure(figsize=(6, 4))
plt.plot(fpr, tpr, label=f"Logistic Regression (AUC = {roc_auc_score(y_test, y_prob):.3f})")
plt.plot([0, 1], [0, 1], "--", label="Random baseline")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curve")
plt.legend()
plt.show()

## 10. Compare actual and predicted labels

In [ ]:
comparison = pd.DataFrame({
    "Actual": y_test.reset_index(drop=True),
    "Predicted": pd.Series(y_pred)
})
display(comparison.head(20))

## 11. Predict one held-out test record
This demonstrates prediction on a row not used to fit the model. It is not a new patient's medical assessment.

In [ ]:
sample = X_test.iloc[[0]]
sample_prediction = model.predict(sample)[0]
sample_probability = model.predict_proba(sample)[0, 1]

print("Predicted class:", sample_prediction)
print("Estimated model probability for class 1:", round(sample_probability, 4))
print("Actual test label:", y_test.iloc[0])

## 12. Predict a manually entered example (optional)
Replace the example values with a row containing all 15 expected feature names. These illustrative values are for software testing only, not clinical guidance.

In [ ]:
new_patient = pd.DataFrame([{
    "male": 1,
    "age": 45,
    "education": 2,
    "currentSmoker": 0,
    "cigsPerDay": 0,
    "BPMeds": 0,
    "prevalentStroke": 0,
    "prevalentHyp": 0,
    "diabetes": 0,
    "totChol": 200,
    "sysBP": 120,
    "diaBP": 80,
    "BMI": 25,
    "heartRate": 75,
    "glucose": 85
}])

new_patient = new_patient[expected_features]
pred = model.predict(new_patient)[0]
prob = model.predict_proba(new_patient)[0, 1]
print("Predicted class:", pred)
print("Estimated class-1 probability:", round(prob * 100, 2), "%")

## 13. Conclusion
This project demonstrates loading a CSV, verifying data, handling missing values and duplicates, splitting data, fitting a Logistic Regression pipeline, evaluating predictions, and saving the model.

Report the metric values produced by your own Colab run; do not copy or invent an accuracy value. This model is educational only and is not validated for clinical use.